This notebook allows to download all SWOT-mission frames within the specified AOI using ``earthaccess`` package

***

__author__ = "Miguel González Jiménez, migj" <br>
__maintainer__ = "Miguel González Jiménez, migj"<br>
__email__ = "mgonzalez.j@gmv.com"

***

In [1]:
import os

import xarray as xr
import pandas as pd
import geopandas as gpd
import rioxarray as riox

import numpy as np

import earthaccess

import SWOT as swot

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from tqdm.notebook import tqdm

### 1. Authentification in earthaccess.
You need to have an account in https://urs.earthdata.nasa.gov/

In [4]:
earthaccess.login()

### 2. Delineating the AOI
As that coinciding with the extension of VIIRS' frames

In [ ]:
viirs_path = r"path_to_one_random_VIIRS_frame.tif"
viirs_example = riox.open_rasterio(viirs_path).sel(band=1)

SS_AOI = swot.get_bbox(viirs_example)

### 3. Retrieving Data
Retrieved data and creating a database for frames managment

In [7]:
# Retrieves granule from the day we want, in this case by passing to `earthdata.search_data` function the data collection shortname, temporal bounds, and for restricted data one must specify the search count
raster_results = earthaccess.search_data(short_name = 'SWOT_L2_HR_Raster_D', # 'SWOT_L2_HR_Raster_2.0', 'SWOT_L2_HR_PIXC_2.0', 'SWOT_L2_HR_RiverSP_2.0
                                        temporal = ('2012-01-01 00:00:00', '2025-12-31 23:59:59'),
                                        bounding_box=swot.get_swot_bbox_format(SS_AOI)
                                        )
df_results = pd.DataFrame(raster_results)
df_results['object'] = raster_results
df_results['name'] = df_results['object'].apply(lambda x: x.get('meta')['native-id'])
df_results['Scene'] = df_results['umm'].apply(lambda x: x['AdditionalAttributes'][0]['Values'][0])
df_results['Cycle'] = df_results['umm'].apply(lambda x: x['SpatialExtent']['HorizontalSpatialDomain']['Track']['Cycle'])
df_results['Pass'] = df_results['umm'].apply(lambda x: x['SpatialExtent']['HorizontalSpatialDomain']['Track']['Passes'][0]['Pass'])
df_results['Tiles'] = df_results['umm'].apply(lambda x: x['SpatialExtent']['HorizontalSpatialDomain']['Track']['Passes'][0]['Tiles'])
df_results['StartTime'] = df_results['umm'].apply(lambda x: x['TemporalExtent']['RangeDateTime']['BeginningDateTime'])

df_results['utm_zone'] = df_results['meta'].apply(lambda x: x['native-id'].split('_')[5])
df_results['valid_utm'] = df_results['utm_zone'].apply(lambda x: np.invert(any([i in x for i in ['60','01']])))
df_results['resolution'] = df_results['name'].apply(lambda x: x.split('_')[4])

# Some frames has a UTM projection not valid according to the coordinates of specified AOI - will be removed
df_results_clean = df_results.loc[df_results.valid_utm]
df_results_clean_100 = df_results_clean.loc[df_results_clean['resolution']== '100m']

# Retaining only frames of 100m spatial resolution
df_results_clean_100['URL'] = df_results_clean_100['umm'].apply(swot.get_url)

Granules found: 4228


In [8]:
print(f"Former df: {df_results_clean.shape[0]}")
print(f"Only 100m df: {df_results_clean_100.shape[0]}")

Former df: 3227
Only 100m df: 1611


In [ ]:
for _,i in tqdm(df_results_clean_100.sort_values('StartTime', ascending=False).iterrows(), unit='file'):
    folder_out = Path(r"folder_to_store_SWOT_100m_frames")
    swot.get_file(i.URL, folder_out)